In [ ]:
import pandas as pd
import numpy as np
from plots import plot_appendix, plot_main_paper
from kcit import kci_test
from scipy.special import expit # expit applies sigmoid to a Series

In [2]:
RESULTS_DIR = 'kcit_randomized_graph_results'

# Simulation 6 Experiments

In [3]:
def generate_sgc_datasets(n,
                          true_upcoding_rates: list[float], true_downcoding_rates: list[float],
                          genuine_modifications_a_1_on_xstar: list[float], genuine_modifications_a_0_on_xstar: list[float]):
    """
    n = number of features (X*'s)
    assuming just 1 downstream variable Y for now, I believe this is WLOG
    all other args are lists of values, 1 list entry per feature (X*) """

    # Load dataset
    df = pd.read_excel('./datasets/default_of_credit_card_clients.xls', skiprows=1)
    df = df.sample(frac=1).reset_index(drop=True)

    # Drop rows that are not needed
    df = df[['SEX', 'EDUCATION', 'MARRIAGE', 'AGE']]

    # Convert sex to binary variable
    df['SEX'] = df['SEX'] - 1

    # Convert marriage to binary variable
    df['MARRIAGE'] = np.where(df['MARRIAGE'] > 1, 1, 0)

    # Convert education to binary variable
    df['EDUCATION'] = np.where(df['EDUCATION'] > 2, 1, 0)

    # Min-max scale the data
    df['AGE'] = (df['AGE'] - df['AGE'].min()) / (df['AGE'].max() - df['AGE'].min())
    
    # Keeping it simple by including no selection bias in agent_prob for now
    agent_prob = 0.3
    df['AGENT'] = np.random.binomial(1, agent_prob, len(df))

    causal_effect_of_x_on_y_list = []
    y_prob = 0.05 \
                    + df['EDUCATION']*0.05 \
                    + df['MARRIAGE']*df['SEX']*0.3 \
                    + np.square(df['AGE'])*0.1
    for i in range(n):
        variable_name = f"X{i}"

        x_prob = 0.05 \
                        + df['EDUCATION']*0.05 \
                        + df['MARRIAGE']*df['SEX']*0.3 \
                        + np.square(df['AGE'])*0.1 \
                        + df['AGENT'] * genuine_modifications_a_1_on_xstar[i] + (1-df['AGENT']) * genuine_modifications_a_0_on_xstar[i]
        df[variable_name] = np.random.binomial(1, expit(x_prob), len(df))

        # Strategically misreport the dataset (misreported employment status)
        prob_required_for_upcoding_rate = ((x_prob / (1-true_upcoding_rates[i])) - x_prob) / (1 - x_prob)
        prob_required_for_downcoding_rate = (((1 - x_prob) / (1-true_downcoding_rates[i])) - (1 - x_prob)) / x_prob

        df[variable_name] = (df[variable_name] + df['AGENT']*(1-df[variable_name]) * np.random.binomial(1, expit(prob_required_for_upcoding_rate),  len(df))
                            - df[variable_name]*(1-df['AGENT']) * np.random.binomial(1, expit(prob_required_for_downcoding_rate), len(df)))
        
        values = np.array([0.0, 0.0, 0.0, 0.0, 0.90, 1.0, 1.1, 1.2]) # we have to take large values here because this is ultimately going through a sigmoid??
        causal_effect = np.random.choice(values)
        causal_effect_of_x_on_y_list.append(causal_effect)
        y_prob += df[variable_name] * causal_effect
        
    # muskaan comment: te i.e., x1_causal_effect_on_y, is beta(x) in the paper
    df['Y'] = np.random.binomial(1, expit(y_prob), len(df)) # expit applies sigmoid to a Series

    return df, causal_effect_of_x_on_y_list

# Sensitivity Analysis Results On Semi-Synthetic Loan Data

### Vary Genuine Adaptation of A onto X2*

In [4]:
num_sims = 2
n = 10 # n = number of features (HCCs) in this simulation

# Dataframes to keep track of results
dfs = []

num_correct = 0

# Perform a few simulations for each method
for sim in range(num_sims):
    # Set the random seed
    np.random.seed(sim)

    # Generate dataset for simulation
    df, causal_effects_of_x_on_y = generate_sgc_datasets(n, [0.1, 0.2, 0.3, 0.0, 0.05, 0.06, 0.07, 0.04, 0.03, 0.3, 0.4, 0.3, 0.1],
                    [0.04, 0.1, 0.2, 0.3, 0.0, 0.05, 0.06, 0.4, 0.03, 0.6, 0.1, 0.4, 0.3, 0.1],  
                    [ 0.04, 0.03, 0.6, 0.1, 0.2, 0.3, 0.0, 0.05, 0.01, 0.07, 0.4, 0.3, 0.1],
                    [ 0.3, 0.05, 0.06, 0.17, 0.03, 0.2, 0.1, 0.2, 0.3, 0.0, 0.4, 0.3, 0.1])
    
    p_values = kci_test(df, n, ['AGENT', 'EDUCATION', 'SEX', 'MARRIAGE', 'AGE'])
    accurate = []

    for i in range(n):
        true_ce = causal_effects_of_x_on_y[i] > 0
        cond_dep = (p_values[i] < 0.05) # low p-value means there is a conditional causal effect
        if(true_ce == cond_dep):
            num_correct += 1
            accurate.append(1)
        else:
            accurate.append(0)

    df_sim = pd.DataFrame({
        "simulation_number": sim,
        "hcc_number": range(len(causal_effects_of_x_on_y)),
        "causal_effect_x_on_y": causal_effects_of_x_on_y,
        "p_value": p_values,
        "accurate": accurate,
    })

    dfs.append(df_sim)

num_datapoints = num_sims * n
accuracy = num_correct / num_datapoints
print(f"accuracy is {accuracy}")

df = pd.concat(dfs, ignore_index=True)
print(df)

accuracy is 0.95
    simulation_number  hcc_number  causal_effect_x_on_y       p_value  \
0                   0           0                   0.0  2.331440e-03   
1                   0           1                   0.9  7.948932e-05   
2                   0           2                   0.9  2.129240e-04   
3                   0           3                   1.1  8.556062e-07   
4                   0           4                   0.0  7.897335e-01   
5                   0           5                   1.2  1.032420e-03   
6                   0           6                   0.9  3.529243e-07   
7                   0           7                   1.1  3.763141e-02   
8                   0           8                   0.0  1.359669e-01   
9                   0           9                   1.2  1.271959e-03   
10                  1           0                   1.0  2.339418e-11   
11                  1           1                   0.0  3.260293e-01   
12                  1           2 